In [2]:
from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers")


*We are importing the `load_dataset` function from the Hugging Face `datasets` library. This library is the absolute industry standard right now for downloading and processing NLP data efficiently.

Why you used this method? ->  Hugging Face `datasets` library uses Apache Arrow under the hood. This means it uses memory-mapping—allowing you to load huge datasets that are larger than your computer's RAM without crashing your system. It's highly optimized. 

### **The Output**

Once this code finishes running, the `dataset` variable will hold a Hugging Face `DatasetDict` object. Think of it like a highly optimized dictionary or a supercharged pandas DataFrame. It will contain the raw text data (specifically the abstracts and titles of the research papers). This is the exact raw material we need to start cleaning and feeding into our sentence transformers to create vector embeddings .

In [3]:
import pandas as pd
df = dataset["train"].to_pandas()
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'], dtype='str')

In [4]:
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


df.head(50000) — deterministic slice, not a random sample
What's happening: df.head(50000) takes the first 50,000 rows in whatever order the dataset arrives in. arXiv IDs are date-based, so it's plausible the sample skews toward earlier submissions.

random_state=42 makes the sample exactly reproducible, so re-running the notebook from scratch reproduces the same 50,000 rows and our cached embeddings/index stay valid.

In [ ]:
df = df[['title', 'abstract']]
# Before: deterministic, order-dependent
# df = df.head(50000)

# After: random and reproducible
df = df.sample(n=50000, random_state=42)
df.isnull().sum()

# Step 1 —> drop if any one title , abstract is empty but i choosed to not do this here , as title may be missing but abstract is there
# df.dropna(subset=["title","abstract"])  # here we can see there are no empty values in either col

#df = df.dropna(subset=["title", "abstract"], how="all") # it drops a row only when both these col are empty

# Step 2 — Drop duplicates on abstract text
df = df.drop_duplicates(subset=["abstract"])

title       0
abstract    0
dtype: int64

Those `Unnamed` columns are just leftover artifacts—usually from when the original dataset creator saved their file as a CSV without ignoring the index. They are basically just old row numbers that got imported as actual data.

Dropping them and keeping only the `title` and `abstract` for two main reasons:

1. **Semantic Irrelevance (Removing Noise):** Since we are building a NLP pipeline to create vector embeddings, our sentence transformer only cares about the actual meaning of words. Row numbers (0, 1, 2, 3...) have absolutely zero semantic value. If we leave them in and accidentally feed them to our model later, it just adds useless noise to your embeddings.
2. **Memory Optimization:** we are working with a massive dataset of over 117,000 rows. Storing redundant integer columns wastes your computer's RAM. Slicing the DataFrame down to just the essential text makes it lighter and speeds up the preprocessing you are about to do.

To merge the title and abstract into a single, context-rich text block so sentence transformer can generate one comprehensive embedding per research paper.

In [ ]:
df["paper_text"] = df["title"] + " " + df["abstract"]
df

#if we drop by only abtract is empty so make paper_text by fill na to fill missing values on title is empty
# df["paper_text"] = (
#     df["title"].fillna("")
#     + " "
#     + df["abstract"]
# ).str.strip()

,title,abstract,paper_text
107644,SVD Perspectives for Augmenting DeepONet Flexi...,Deep operator networks (DeepONets) are power...,SVD Perspectives for Augmenting DeepONet Flexi...
33964,Towards robust audio spoofing detection: a det...,"Automatic speaker verification, like every o...",Towards robust audio spoofing detection: a det...
3137,Guided Random Forest in the RRF Package,Random Forest (RF) is a powerful supervised ...,Guided Random Forest in the RRF Package Rand...
33168,Best Arm Identification in Generalized Linear ...,"Motivated by drug design, we consider the be...",Best Arm Identification in Generalized Linear ...
20962,Conditional Affordance Learning for Driving in...,Most existing approaches to autonomous drivi...,Conditional Affordance Learning for Driving in...
...,...,...,...
76007,Residual-Aided End-to-End Learning of Communic...,Leveraging powerful deep learning techniques...,Residual-Aided End-to-End Learning of Communic...
72993,One-Class Classification: A Survey,One-Class Classification (OCC) is a special ...,One-Class Classification: A Survey One-Class...
59276,It's Time to Play Safe: Shield Synthesis for T...,Erroneous behaviour in safety critical real-...,It's Time to Play Safe: Shield Synthesis for T...
54357,Deep Learning for Frame Error Prediction using...,We demonstrate a first example for employing...,Deep Learning for Frame Error Prediction using...


* **What the code is doing:** It takes the string from the `title` column, adds a single space (`" "`), and appends the string from the `abstract` column. It saves this combined text into a brand new column called `paper_text`.
* **Why we do this :** Sentence transformers convert text into mathematical vectors (embeddings) based on context.
* embedding just for the title, it might lack depth.
* embedding just for the abstract, it might miss the core, punchy keywords the author put in the title.
* If we embed them separately, our vector database has to do twice the work during a semantic search.

In [25]:
df['paper_text'].head()

107644    SVD Perspectives for Augmenting DeepONet Flexi...
33964     Towards robust audio spoofing detection: a det...
3137      Guided Random Forest in the RRF Package   Rand...
33168     Best Arm Identification in Generalized Linear ...
20962     Conditional Affordance Learning for Driving in...
Name: paper_text, dtype: str

### **The Objective**

To extract and preview our newly created `paper_text` column as a cleanly formatted, 2D table rather than a raw, 1D list of text.

In [8]:
df[["paper_text"]].head()


,paper_text
107644,SVD Perspectives for Augmenting DeepONet Flexi...
33964,Towards robust audio spoofing detection: a det...
3137,Guided Random Forest in the RRF Package Rand...
33168,Best Arm Identification in Generalized Linear ...
20962,Conditional Affordance Learning for Driving in...



### **The Breakdown & Explanation**

In pandas, how you use brackets completely changes the data structure you get back:

* **Single Brackets (`df["paper_text"]`):** This returns a **pandas Series**. Think of a Series as a 1D array or a single raw column of data. It doesn't have the nice visual table formatting in Jupyter; it just prints out as a plain text list.
* **Double Brackets (`df[["paper_text"]]`):** The inner brackets create a list of column names (even if it's just one name), and the outer brackets tell pandas to extract that list. This returns a **pandas DataFrame**. Even though it only contains one column, pandas still treats it as a 2D mathematical table.

**Summary:** "Single brackets extract a 1D Series, while double brackets extract a 2D DataFrame. I used double brackets here purely for better visualization in the notebook before passing the text to the sentence transformer."

In [9]:
print(df['paper_text'].iloc[0])

SVD Perspectives for Augmenting DeepONet Flexibility and
  Interpretability   Deep operator networks (DeepONets) are powerful architectures for fast and
accurate emulation of complex dynamics. As their remarkable generalization
capabilities are primarily enabled by their projection-based attribute, we
investigate connections with low-rank techniques derived from the singular
value decomposition (SVD). We demonstrate that some of the concepts behind
proper orthogonal decomposition (POD)-neural networks can improve DeepONet's
design and training phases. These ideas lead us to a methodology extension that
we name SVD-DeepONet. Moreover, through multiple SVD analyses, we find that
DeepONet inherits from its projection-based attribute strong inefficiencies in
representing dynamics characterized by symmetries. Inspired by the work on
shifted-POD, we develop flexDeepONet, an architecture enhancement that relies
on a pre-transformation network for generating a moving reference frame and
isolat

### **The Objective**

To load the specific neural network model that will actually perform the core task: converting cleaned text into mathematical vector embeddings.

In [10]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)

In [11]:
print(type(model))

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


### **The Objective**

To isolate a single test string from our dataset. We do this to run a quick, small-scale test on our sentence transformer before committing the time and compute power to encode all 50000 rows.

In [12]:
sample_text = df["paper_text"].iloc[0]
sample_text

"SVD Perspectives for Augmenting DeepONet Flexibility and\n  Interpretability   Deep operator networks (DeepONets) are powerful architectures for fast and\naccurate emulation of complex dynamics. As their remarkable generalization\ncapabilities are primarily enabled by their projection-based attribute, we\ninvestigate connections with low-rank techniques derived from the singular\nvalue decomposition (SVD). We demonstrate that some of the concepts behind\nproper orthogonal decomposition (POD)-neural networks can improve DeepONet's\ndesign and training phases. These ideas lead us to a methodology extension that\nwe name SVD-DeepONet. Moreover, through multiple SVD analyses, we find that\nDeepONet inherits from its projection-based attribute strong inefficiencies in\nrepresenting dynamics characterized by symmetries. Inspired by the work on\nshifted-POD, we develop flexDeepONet, an architecture enhancement that relies\non a pre-transformation network for generating a moving reference fra

### **The Breakdown & Explanation**

* **Summary:** "Before processing a massive batch of data through a neural network, it is an industry best practice to isolate a single sample (like index 0) and pass it through the pipeline to verify the logic and output shape. It saves hours of potential debugging."


### **The Objective**

To remove structural formatting garbage (specifically newline characters) from our text so the transformer reads it as one smooth, continuous paragraph rather than broken, fragmented lines.


In [ ]:
# df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
# df["paper_text"] = df["paper_text"].str.strip()
# 

# Step 3 :Fix whitespace 
df["paper_text"] = (
    df["paper_text"]
    .str.replace(r"\s+", " ", regex=True)  # collapse ALL whitespace to single space
    .str.strip() # remove leading/trailing whitespace
)

# Step 4 : now when extra spaces are removed we can remove short papers that have no real data 
df = df[df["paper_text"].str.len() > 30]

# Step 5 : Reset index to start with 0 so embeddings in numpy array and dataframe stays consistent with each other
df = df.reset_index(drop=True)

### **The Breakdown & Explanation**

* **`.str.replace("\n", " ")`**:  pandas look inside every single row of our text column, find the exact invisible character `\n` (which represents a line break or "Enter" key press), and replace it with a standard blank space.

2. **Line 2 (`.str.strip()`):** This removes any completely useless "leading" or "trailing" whitespace.

* **`regex=False`**: This is a performance optimizer. By setting this to False, we are telling pandas, "Don't treat this as a complex Regular Expression search, just look for the literal characters `\n`." Since you have 117,000+ rows, this makes the execution significantly faster.

*so while transformers understand linguistics perfectly, text extracted from academic PDFs often contains arbitrary line breaks depending on column width. If a sentence breaks as 'machine \n learning', it might be tokenized weirdly. Replacing `\n` with a space ensures structural continuity for the attention mechanisms."


In [14]:
sample_text = df["paper_text"].iloc[0]
sample_text

"SVD Perspectives for Augmenting DeepONet Flexibility and   Interpretability   Deep operator networks (DeepONets) are powerful architectures for fast and accurate emulation of complex dynamics. As their remarkable generalization capabilities are primarily enabled by their projection-based attribute, we investigate connections with low-rank techniques derived from the singular value decomposition (SVD). We demonstrate that some of the concepts behind proper orthogonal decomposition (POD)-neural networks can improve DeepONet's design and training phases. These ideas lead us to a methodology extension that we name SVD-DeepONet. Moreover, through multiple SVD analyses, we find that DeepONet inherits from its projection-based attribute strong inefficiencies in representing dynamics characterized by symmetries. Inspired by the work on shifted-POD, we develop flexDeepONet, an architecture enhancement that relies on a pre-transformation network for generating a moving reference frame and isola

### **The Objective**

To execute the core NLP task: passing your test text through the neural network to generate a mathematical vector, and then verifying its structure to ensure it is ready for a vector database.


In [15]:
embedding = model.encode(sample_text)
print(type(embedding))
print(embedding.shape)

<class 'numpy.ndarray'>
(384,)


### **The Breakdown & Explanation**

* **`print(type(embedding))`**: This checks the data type. It returns `<class 'numpy.ndarray'>`.
* **Interview Context:** Why does this matter? Because standard Python lists are too slow for heavy math. NumPy arrays are highly optimized in C, and almost all vector databases (like ChromaDB, Pinecone, or Milvus) and similarity algorithms (like Cosine Similarity) strictly require NumPy arrays to run semantic searches efficiently.


* **`print(embedding.shape)`**: This checks the dimensions of the array. It returns `(384,)`.
* **Interview Context:** If they ask, "How do you know the model worked correctly?" you point to this. You specifically loaded a model designed to output a 384-dimensional vector. Because the shape is exactly `(384,)`, it confirms the model successfully compressed the text's meaning into exactly 384 floating-point numbers.



### **The Output**

The output confirms two critical things: you have a highly optimized NumPy array, and it has the exact mathematical shape (384 dimensions) required for our downstream semantic search.


In [16]:
embedding[:56]

array([-0.08374168, -0.05528529,  0.01438526,  0.0069977 ,  0.00071513,
        0.0359526 , -0.09253504, -0.03199809,  0.07097851, -0.06945141,
       -0.00987536, -0.01322999, -0.13026695,  0.04086325, -0.09236524,
       -0.03323522,  0.06573877,  0.14673965, -0.14218394, -0.01326887,
        0.02968766,  0.00022744,  0.02444846,  0.00256727,  0.08639959,
        0.03091933, -0.02631026, -0.02217486,  0.00678797, -0.05297283,
        0.05825306, -0.01205868, -0.10175437, -0.02051953, -0.0722219 ,
        0.11313103,  0.03504826,  0.08751994, -0.09865805, -0.0307427 ,
        0.03525425,  0.08763696,  0.03622475,  0.05960535,  0.02863871,
        0.04772096,  0.00652322, -0.11577681, -0.0051812 ,  0.06743818,
       -0.05297126,  0.02869037,  0.04977187,  0.05859168,  0.0283855 ,
       -0.01805156], dtype=float32)

In [17]:
sample_embedding = model.encode(
    df["paper_text"].head(5).to_list()
)

print(sample_embedding.shape)

(5, 384)



If the output shape was just `(5,)`, it would mean the AI compressed each of our 5 research papers into a *single number*. We can't perform Cosine Similarity on a single number we need an actual vector (a coordinate in multi-dimensional space) to measure an angle!

We fed the model a list of 5 separate research papers (`.head(5)`). The model processed them as a batch, doing the exact same thing it did to your single test sample:

* Paper 1 ➔ Converted into 384 features
* Paper 2 ➔ Converted into 384 features
* Paper 3 ➔ Converted into 384 features
* Paper 4 ➔ Converted into 384 features
* Paper 5 ➔ Converted into 384 features

NumPy automatically stacks these together into a 2D matrix. So, `(5, 384)` simply means our array has **5 rows** ( papers) and **384 columns** (the mathematical embedding for each paper).

When we run our Cosine Similarity function, it will take the 384 numbers from one row, compare them mathematically to the 384 numbers of another row, and calculate the angle between them to see how semantically similar those two papers are.

### **The Objective**

To load the mathematical function that will calculate how closely related your research papers are by measuring the angles between their 384-dimensional embeddings.


In [18]:
import sklearn
from sklearn.metrics.pairwise import cosine_similarity

### **The Breakdown & Explanation**

* **`sklearn.metrics.pairwise`**: You are diving into `scikit-learn` (the standard Python library for classical machine learning). The `pairwise` module contains tools to calculate the distances or similarities between arrays of data.
* **`cosine_similarity`**: This specific function measures the cosine of the angle between two vectors.
* **Interview Context:** This is a guaranteed interview question. If an interviewer asks, *"Why use Cosine Similarity instead of standard Euclidean distance (straight-line distance) for text embeddings?"*, here is your answer:
> "Euclidean distance measures magnitude (how long a document is), but Cosine Similarity measures orientation (the actual semantic direction). If Paper A is a short summary about neural networks and Paper B is a 50-page thesis on neural networks, their vectors will point in the exact same direction, giving them a high cosine similarity score (near 1.0). Euclidean distance would falsely flag them as different simply because Paper B has more words and a longer vector magnitude."


In [19]:
similarity = cosine_similarity([sample_embedding[0]], [sample_embedding[0]])
print(similarity)

[[1.]]


### **The Fix**

You need to pass the embeddings as 2D arrays (matrices), not 1D arrays (flat lists). The easiest way to fix this is to wrap your slices in an extra set of square brackets.


*(Alternatively, you could also write it using NumPy's reshape method: `sample_embedding[0].reshape(1, -1)`, but the extra brackets do the exact same thing and are easier to read!)*

---

### **What went wrong (The `ValueError`)**

This is a very common `scikit-learn` error.

Think back to the shape of your `sample_embedding`. It was `(5, 384)`—a 2D matrix.
When you sliced it using `sample_embedding[0]`, you extracted just the very first row. NumPy automatically flattened that row into a 1D array of shape `(384,)`.

The `cosine_similarity` function is designed to compare *batches* of data. It strictly expects a 2D array as its input (meaning it wants a "list of papers", even if that list only contains one paper). Because you handed it a 1D array, it threw a `ValueError` saying it expected 2D data but got 1D. By wrapping it in brackets `[sample_embedding[0]]`, you temporarily turn it back into a 2D array of shape `(1, 384)`.

---

### **What this code is actually doing**

* **The Math:** You are telling the computer to measure the angle between the vector of Paper 1 and the vector of... Paper 1.
* **The Expected Output:** Because you are comparing a paper to its exact self, the two vectors point in the exact same mathematical direction. The angle between them is 0 degrees.
* **The Result:** The cosine of 0 is 1. When you run the fixed code, it will print out `[[1.]]` (or `[[1.00000000e+00]]`), which translates to a **100% semantic match**.

The next logical step is to compare `[sample_embedding[0]]` against `[sample_embedding[1]]` to see how mathematically similar Paper 1 and Paper 2 are!

In [20]:
similarity = cosine_similarity(sample_embedding[0].reshape(1, -1), sample_embedding[1].reshape(1, -1))
print(similarity)

[[0.27217373]]



### **The Breakdown & Explanation**

**1. The `.reshape(1, -1)` trick:**
In the last step, you got an error because `cosine_similarity` strictly requires 2D arrays (matrices), but `sample_embedding[0]` is a flat 1D array.
While wrapping it in brackets `[sample_embedding[0]]` worked, using `.reshape(1, -1)` is the **official, industry-standard NumPy way** to fix this.

* The `1` tells NumPy: "Make this exactly 1 row."
* The `-1` tells NumPy: "Calculate the number of columns automatically based on the data." (It automatically figures out there are 384 columns).

**2. What does `0.27217373` mean?**
You just mathematically compared Paper 1 against Paper 2!
Cosine similarity returns a score between **-1.0 and 1.0**.

* **1.0** = Exact same meaning (like when you compared Paper 1 to itself).
* **0.0** = Completely unrelated (perpendicular vectors).
* **-1.0** = Exact opposite meaning.

A score of **0.27217373** means these two research papers have a **moderate-to-low similarity**. Because they are both Machine Learning papers from ArXiv, they likely share some common academic vocabulary (like "data", "algorithm", "results"), which keeps the score above zero, but their actual core research topics are distinctly different.


In [21]:
# 1. Run the import again to fix the NameError
from sklearn.metrics.pairwise import cosine_similarity

# 2. Run your loop
for i in range(1, 5):
    sim = cosine_similarity(sample_embedding[0].reshape(1, -1), sample_embedding[i].reshape(1, -1))
    print(f"Similarity with Paper {i}: {sim[0][0]}")

Similarity with Paper 1: 0.2721737325191498
Similarity with Paper 2: 0.2350483387708664
Similarity with Paper 3: 0.07694458961486816
Similarity with Paper 4: 0.30208730697631836



### **The Objective**
To unleash the neural network on your *entire* dataset. Instead of testing just 1 or 5 papers, this step converts all of our cleaned research papers into 384-dimensional mathematical vectors so they can be stored and searched.

In [ ]:
# embedding = model.encode(df["paper_text"].to_list(), batch_size=32, show_progress_bar=True)
# dont direclty run this statement as running multiple times if embeddeds were already there in data folder so dont waste time here

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]


### **The Explanation**

* **`df["paper_text"].to_list()`**: The `sentence-transformers` neural network is not designed to read pandas DataFrame columns directly. It expects a standard Python list, text or images `. `.to_list()` quickly rips all the text out of your dataframe and packages it into a massive Python list of strings.
* **`batch_size=32`**: This tells the model to grab 32 papers, process them, save the vectors, and then grab the next 32.
* **`show_progress_bar=True`**: Because processing thousands of papers takes heavy math, it takes time. This turns on a visual loading bar so you know your code hasn't frozen.

### **The Output**

While it runs, you see the progress bar: `1/469 [00:03<26:19, 3.37s/it]`

* **`1/469`**: The model has divided your dataset into 469 batches (chunks of 32 papers).
* **`<26:19`**: It estimates it will take about 26 minutes to finish all the math.
* **When it reaches 100%:** The `embedding` variable will hold a gigantic 2D NumPy matrix containing the vector for every single paper. If you ran this on 15,000 papers, the shape will be exactly `(15000, 384)`.

### **Why we did it (Interview Context)**

If an interviewer asks, *"Why did you explicitly set a `batch_size` instead of just passing the whole list?"* **Your Answer:** "To prevent **Out-Of-Memory (OOM) errors**. You cannot feed 15,000+ massive text documents into a neural network simultaneously; the computer's RAM (or GPU VRAM) will instantly overload and crash the program. By setting `batch_size=32`, I engineered the pipeline to process the data in manageable chunks. It is a standard industry practice to balance processing speed with memory safety."

In [23]:
import numpy as np
import os
index_path = "../data/arxiv_embeddings.npy"

if os.path.exists(index_path):
    print("Loading saved embeddings")
    embeddings = np.load(index_path)
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save(index_path, embeddings)
    print("Embeddings saved successfully!")
# This saves your 26-minute calculation instantly to a file in your folder

print("Embeddings successfully saved to disk!")

Loading saved embeddings
Embeddings successfully saved to disk!


In [24]:
# Save the cleaned dataframe so we can map the math back to actual paper titles
df.to_csv("../data/cleaned_arxiv_papers.csv", index=False)
print("Cleaned DataFrame saved successfully!")

Cleaned DataFrame saved successfully!
